# 🧠 02. RGB Ekstraksi: Ekstraksi Fitur CNN (Layer `svm_features`)

Notebook ini menggunakan model CNN yang telah dilatih pada `01_RGB_Training.ipynb` sebagai **feature extractor**.
Fitur representasi tingkat tinggi berdimensi 512 diekstrak dari layer `svm_features` (GlobalAveragePooling2D) dan disimpan ke disk dalam format `.npz`.


In [1]:
import os
import sys
import numpy as np
import tensorflow as tf


ARTIFACT_DIR = 'cifar10_artifacts/rgb'
cnn_model_path = os.path.join(ARTIFACT_DIR, 'cnn_model.keras')
print(f"Memuat model CNN terlatih dari: {cnn_model_path}")
cnn_model = tf.keras.models.load_model(cnn_model_path)

from tensorflow.keras.models import Model

def load_cifar10_data(load_train=True, load_test=True):
    (X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
    if load_train:
        X_train = X_train.astype('float32') / 255.0
        y_train = y_train.flatten()
    else:
        X_train, y_train = None, None
    if load_test:
        X_test = X_test.astype('float32') / 255.0
        y_test = y_test.flatten()
    else:
        X_test, y_test = None, None
    return X_train, y_train, X_test, y_test

def extract_cnn_features(model, X_data, batch_size=128, use_tta=True):
    feature_layer = model.get_layer('svm_features')
    extractor = tf.keras.models.Model(inputs=model.input, outputs=feature_layer.output)
    print(f"Feature Extractor Layer : {feature_layer.name} | Output Shape : {feature_layer.output_shape}")
    total_samples = len(X_data)
    features = []
    for i in range(0, total_samples, batch_size):
        end = min(i + batch_size, total_samples)
        batch = X_data[i:end]
        f1 = extractor.predict(batch, batch_size=batch_size, verbose=0)
        if use_tta:
            f2 = extractor.predict(np.flip(batch, axis=2), batch_size=batch_size, verbose=0)
            feats = 0.5 * (f1 + f2)
        else:
            feats = f1
        features.append(feats)
    features_arr = np.vstack(features)
    print(f"[EKSTRAKSI SELESAI] Extracted Features Shape: {features_arr.shape}")
    return features_arr


Memuat model CNN terlatih dari: cifar10_artifacts/rgb\cnn_model.keras


## 📥 1. Memuat Data & Mengekstrak Vektor Fitur CNN


In [2]:
X_train, y_train, X_test, y_test = load_cifar10_data()

print("Mengekstrak fitur CNN dari data training (50.000 sampel)...")
X_train_features = extract_cnn_features(cnn_model, X_train, batch_size=128)

print()
print("Mengekstrak fitur CNN dari data testing (10.000 sampel)...")
X_test_features = extract_cnn_features(cnn_model, X_test, batch_size=128)

print()
print(f"[SELESAI] X_train_features shape: {X_train_features.shape}")
print(f"[SELESAI] X_test_features shape : {X_test_features.shape}")


Mengekstrak fitur CNN dari data training (50.000 sampel)...
Feature Extractor Layer : svm_features | Output Shape : (None, 512)
[EKSTRAKSI SELESAI] Extracted Features Shape: (50000, 512)

Mengekstrak fitur CNN dari data testing (10.000 sampel)...
Feature Extractor Layer : svm_features | Output Shape : (None, 512)
[EKSTRAKSI SELESAI] Extracted Features Shape: (10000, 512)

[SELESAI] X_train_features shape: (50000, 512)
[SELESAI] X_test_features shape : (10000, 512)


## 💾 2. Menyimpan Vektor Fitur ke Disk (.npz)


In [3]:
train_feat_path = os.path.join(ARTIFACT_DIR, 'train_features.npz')
test_feat_path  = os.path.join(ARTIFACT_DIR, 'test_features.npz')

np.savez_compressed(train_feat_path, X_train_features=X_train_features, y_train=y_train)
np.savez_compressed(test_feat_path, X_test_features=X_test_features, y_test=y_test)

print(f"[BERHASIL] Fitur training tersimpan: {train_feat_path}")
print(f"[BERHASIL] Fitur testing tersimpan : {test_feat_path}")


[BERHASIL] Fitur training tersimpan: cifar10_artifacts/rgb\train_features.npz
[BERHASIL] Fitur testing tersimpan : cifar10_artifacts/rgb\test_features.npz
